#### Curried Functions

In [1]:
max 4 5
(max 4) 5

Line 2: Redundant bracket
Found:
(max 4) 5
Why not:
max 4 5

5

5

In [2]:
multThree :: (Num a) => a -> a -> a -> a
multThree x y z = x * y * z

In [6]:
multTwoWithFive = multThree 5
multTwoWithFive 2 3

multWithTen = multTwoWithFive 2
multWithTen 2


30

20

In [7]:
compareWithHundred :: (Num a, Ord a) => a -> Ordering
compareWithHundred x = compare 100 x

compareWithHundred 99

Line 2: Eta reduce
Found:
compareWithHundred x = compare 100 x
Why not:
compareWithHundred = compare 100

GT

In [10]:
compareWithHundred :: (Num a, Ord a) => a -> Ordering
compareWithHundred = compare 100

compareWithHundred 99

GT

In [11]:
divideByTwo :: (Floating a) => a -> a
divideByTwo = (/2)

divideByTwo 5

2.5

In [13]:
isUpper :: Char -> Bool
isUpper = (`elem` ['A'..'Z'])

isUpper 'b'

False

In [14]:
applyTwice :: (a -> a) -> a -> a
applyTwice f x = f (f x)

In [16]:
applyTwice (/2) 20

5.0

In [18]:
applyTwice (++ " LOL") "HEY"

"HEY LOL LOL"

In [19]:
applyTwice ("LOL " ++) "HEY"

"LOL LOL HEY"

In [20]:
applyTwice (2:) [1]

[2,2,1]

In [21]:
zipWith' :: (a -> b -> c) -> [a] -> [b] -> [c]
zipWith' _ [] _ = []
zipWith' _ _ [] = []
zipWith' f (x:xs) (y:ys) = f x y : zipWith' f xs ys

In [22]:
zipWith' (+) [1, 2, 3, 4] [10, 20, 30]

[11,22,33]

In [24]:
zipWith' max [1, 4, 2] [2, 3, 4]

[2,4,4]

In [29]:
zipWith' (++) ["one: ", "two: ", "three: "] ["1", "2", "3"]

["one: 1","two: 2","three: 3"]

In [30]:
zipWith' (*) (replicate 5 2) [1..]

[2,4,6,8,10]

In [31]:
zipWith' (zipWith' (*)) [[1,2,3],[3,5,6],[2,3,4]] [[3,2,2],[3,4,5],[5,4,3]] 

[[3,4,6],[9,20,30],[10,12,12]]

In [37]:
flip' :: (a -> b -> c) -> (b -> a -> c) -- same with b -> a -> c
flip' f = g
    where g x y = f y x

In [38]:
flip' zip [1..5] "Hello"

[('H',1),('e',2),('l',3),('l',4),('o',5)]

In [41]:
zipWith (flip' div) [2,2..] [10, 8, 6, 4, 2]

[5,4,3,2,1]

In [43]:
map :: (a -> b) -> [a] -> [b]
map _ [] = []
map f (x:xs) = f x : map f xs

In [44]:
map (+3) [1, 3, 5]

[4,6,8]

In [45]:
map (replicate 2) [3..6]

[[3,3],[4,4],[5,5],[6,6]]

In [46]:
map (map (^2)) [[1,2], [3,4,5,6], [7,8]]

[[1,4],[9,16,25,36],[49,64]]

In [51]:
map fst [(1,2), (3,5)]

[1,3]

In [52]:
filter :: (a -> Bool) -> [a] -> [a]
filter _ [] = []
filter p (x:xs)
    | p x       = x : filter p xs
    | otherwise = filter p xs

In [54]:
filter (>10) [11, 2, 34]
filter even [1..20]

[11,34]

[2,4,6,8,10,12,14,16,18,20]

In [60]:
let notNull x = not (null x) in filter notNull [[1,2,3], [], [], [22, 33, 44]]

[[1,2,3],[22,33,44]]

#### Quicksort with filter

In [61]:
quicksort :: (Ord a) => [a] -> [a]  
quicksort [] = []  
quicksort (x:xs) = 
    let smallerSorted = quicksort (filter (<=x) xs)  
        biggerSorted = quicksort (filter (>x) xs)  
    in  smallerSorted ++ [x] ++ biggerSorted  

In [63]:
largestDivisible :: (Integral a) => a -> a
largestDivisible a = head(filter p [10000,9999..])
    where p x = x `mod` a == 0

In [64]:
largestDivisible 2322
largestDivisible 34

9288

9996

In [75]:
sum (takeWhile (<100) (filter odd (map (^2) [1..] )))
sum (takeWhile (<100) [n^2 | n <- [1..], odd(n^2)])

165

165

In [76]:
chain :: (Integral a) => a -> [a]
chain 1 = [1]
chain n
    | even n = n : chain (n `div` 2)
    | odd n = n : chain (n*3 + 1)

In [77]:
chain 30

[30,15,46,23,70,35,106,53,160,80,40,20,10,5,16,8,4,2,1]

In [78]:
numLongChains :: Int
numLongChains = length (filter isLong (map chain [1..100]))
    where isLong xs = length xs > 15

In [79]:
numLongChains

66

In [85]:
listOfFuncs = map (*) [0..10]
(listOfFuncs !! 4) 5
zipWith' ($) listOfFuncs [1,2,3,4,5]

20

[0,2,6,12,20]

#### Lambdas

In [86]:
numLongChains :: Int
numLongChains = length (filter (\xs -> length xs > 15) (map chain [1..100]))

In [87]:
numLongChains

66

In [89]:
map (\(a,b) -> a + b) [(1,2), (3,4)]

Line 1: Use uncurry
Found:
\ (a, b) -> a + b
Why not:
uncurry (+)

[3,7]

In [90]:
flip' :: (a -> b -> c) -> b -> a -> c
flip' f = \x y -> f y x

flip' div 2 4

Line 2: Redundant lambda
Found:
flip' f = \ x y -> f y x
Why not:
flip' f x y = f y xLine 2: Avoid lambda
Found:
\ x y -> f y x
Why not:
flip f

2

#### Folds

In [99]:
sum' :: (Num a) => [a] -> a
sum' xs = foldl (\acc x -> acc + x) 0 xs

Line 2: Eta reduce
Found:
sum' xs = foldl (\ acc x -> acc + x) 0 xs
Why not:
sum' = foldl (\ acc x -> acc + x) 0Line 2: Avoid lambda
Found:
\ acc x -> acc + x
Why not:
(+)

In [100]:
sum' [1, 2, 3]

6

In [101]:
sum' :: (Num a) => [a] -> a
sum' = foldl (+) 0

sum' [3,4,5]

Line 2: Use sum
Found:
foldl (+) 0
Why not:
sum

12

In [102]:
elem' :: (Eq a) => a -> [a] -> Bool
elem' y ys = foldl (\acc x -> if x == y then True else acc) False ys

elem' 'e' "Hello"

Line 2: Eta reduce
Found:
elem' y ys
  = foldl (\ acc x -> if x == y then True else acc) False ys
Why not:
elem' y = foldl (\ acc x -> if x == y then True else acc) FalseLine 2: Redundant if
Found:
if x == y then True else acc
Why not:
(x == y) || acc

True

In [104]:
map' :: (a -> b) -> [a] -> [b]
map' f xs = foldr (\x acc -> f x : acc) [] xs

Line 2: Eta reduce
Found:
map' f xs = foldr (\ x acc -> f x : acc) [] xs
Why not:
map' f = foldr (\ x acc -> f x : acc) []Line 2: Use map
Found:
foldr (\ x acc -> f x : acc) []
Why not:
map (\ x -> f x)

#### Foldr and Foldl on Infinite Lists

In [5]:
foldr f acc [] = acc
foldr f acc (x:xs) = f x (foldr f acc xs)

Line 1: Use foldr
Found:
foldr f acc [] = acc
foldr f acc (x : xs) = f x (foldr f acc xs)
Why not:
foldr f acc xs = foldr f acc xs

In [10]:
containsThree :: [Int] -> Bool
containsThree = foldr (\x rest -> (x == 3) || rest) False

In [11]:
containsThree [1..]

True

In [12]:
foldl f acc [] = acc
foldl f acc (x:xs) = foldl f (f acc x) xs

In [13]:
containsThreeFoldl :: [Int] -> Bool  
containsThreeFoldl = foldl (\acc x -> if x == 3 then True else acc) False

Line 2: Redundant if
Found:
if x == 3 then True else acc
Why not:
(x == 3) || acc

In [1]:
-- Infinite Loop
-- containsThreeFoldl [1..]

#### STD Library Functions with Folds

In [1]:
maximum' :: (Ord a) => [a] -> a
maximum' = foldr1 (\x acc -> if x > acc then x else acc)

maximum' [2, 3, 45, 23, 453]

Line 2: Use max
Found:
if x > acc then x else acc
Why not:
max x acc

453

In [2]:
reverse' :: [a] -> [a]
reverse' = foldl (\acc x -> x : acc) []

reverse' [1..20]

Line 2: Avoid lambda
Found:
\ acc x -> x : acc
Why not:
flip (:)

[20,19,18,17,16,15,14,13,12,11,10,9,8,7,6,5,4,3,2,1]

In [5]:
product' :: (Num a) => [a] -> a
product' = foldr1 (*)

product' [1..5]

Line 2: Use product
Found:
foldr1 (*)
Why not:
product

120

In [15]:
filter' :: (a -> Bool) -> [a] -> [a]
filter' p = foldr (\x acc -> if p x then x : acc else acc) []

filter' odd [1..20]

[1,3,5,7,9,11,13,15,17,19]

In [19]:
head' :: [a] -> a
head' = foldr1 const
-- head' = foldr1 (\x _ -> x)

head' [2..20]

2

In [20]:
last' :: [a] -> a
last' = foldl1 (\_ x -> x)

head' [2..20]

2

#### Scanl and Scanr

In [22]:
scanl (+) 0 [3, 2, 4, 6]
scanr (+) 0 [3, 2, 4, 6]

[0,3,5,9,15]

[15,12,10,6,0]

In [23]:
scanl1 (\acc x -> if x > acc then x else acc) [3, 2, 4, 11, 2, 45, 23]

Line 1: Use max
Found:
if x > acc then x else acc
Why not:
max x acc

[3,3,4,11,11,45,45]

In [26]:
scanl (flip (:)) [] [3,2,1]

[[],[3],[2,3],[1,2,3]]

In [27]:
sqrtSums :: Int
sqrtSums = length (takeWhile (<1000) (scanl1 (+) (map sqrt [1..]))) + 1

In [28]:
sqrtSums

131

In [31]:
sum (map sqrt [1..131])
sum (map sqrt [1..130])

1005.0942035344083

993.6486803921487

#### Function application with $

In [44]:
infixr 0 $
($) :: (a -> b) -> a -> b
f $ x = f x

In [45]:
sqrt $ 3 + 4 + 9

4.0

In [46]:
map ($ 3) [(4+), (10*), (^2), sqrt]

[7.0,30.0,9.0,1.7320508075688772]

#### Function composition

In [47]:
(.) :: (b -> c) -> (a -> b) -> a -> c
f . g = \x -> f (g x)

Line 2: Avoid lambda
Found:
\ x -> f (g x)
Why not:
f . g

In [48]:
map (\x -> negate (abs x)) [5,-3,-6,7,-3,2,-19,24]

Line 1: Avoid lambda
Found:
\ x -> negate (abs x)
Why not:
negate . abs

[-5,-3,-6,-7,-3,-2,-19,-24]

In [49]:
map (negate . abs) [5,-3,-6,7,-3,2,-19,24]

[-5,-3,-6,-7,-3,-2,-19,-24]

In [50]:
map (\xs -> negate (sum (tail xs))) [[1..5],[3..6],[1..7]]

Line 1: Avoid lambda
Found:
\ xs -> negate (sum (tail xs))
Why not:
negate . sum . tail

<interactive>:1:26: warning: [GHC-63394] [-Wx-partial]
    In the use of ‘tail’ (imported from Prelude, but defined in GHC.Internal.List):
    "This is a partial function, it throws an error on empty lists. Replace it with 'drop' 1, or use pattern matching or 'GHC.Internal.Data.List.uncons' instead. Consider refactoring to use "Data.List.NonEmpty"."

[-14,-15,-27]

In [51]:
map (negate . sum . tail) [[1..5],[3..6],[1..7]]

<interactive>:1:21: warning: [GHC-63394] [-Wx-partial]
    In the use of ‘tail’ (imported from Prelude, but defined in GHC.Internal.List):
    "This is a partial function, it throws an error on empty lists. Replace it with 'drop' 1, or use pattern matching or 'GHC.Internal.Data.List.uncons' instead. Consider refactoring to use "Data.List.NonEmpty"."

[-14,-15,-27]

In [52]:
oddSquareSum :: Integer
oddSquareSum = sum . takeWhile (<10000) . filter odd . map (^2) $ [1..]

oddSquareSum

166650

In [53]:
oddSquareSum :: Integer
oddSquareSum =
    let oddSquares = filter odd $ map (^2) [1..]
        belowLimit = takeWhile (<10000) oddSquares
    in sum belowLimit

oddSquareSum

166650